In [2]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import os
import seaborn as sns
import sys
import glob
import tqdm
plt.style.use('default')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 22, 'legend.facecolor': 'white', 'legend.framealpha': 1, "legend.frameon": 1, "lines.linewidth": 2})

In [3]:
hek_label = pd.read_pickle("/extdata4/baeklab/Hyeonseo/m6A/runs/exp_MRNA/ON0090/ON0090/result/reference_test/dorado_output.pileup.filtered.pkl")

In [4]:
hela_label = pd.read_pickle("/extdata4/baeklab/Hyeonseo/m6A/runs/exp_MRNA/ON0091/ON0091/result/reference_test/dorado_output.pileup.filtered.pkl")

In [5]:
def unique5mer(x):
    return ":".join(x.unique())

In [8]:
sys.path.append("/extdata4/baeklab/Hyeonseo/m6A/modformer")
from utils import utils

In [9]:
refflat_df = utils.parse_refflat_v2()

In [10]:
print(hek_label)

                         ref  pos base  depth   5mer
0             XM_047432085.1   12    A      1  CGACT
1             XM_047432085.1   28    A      2  GGAGG
2             XM_047432085.1   37    A      2  TGAGG
3             XM_047432085.1   42    A      2  CGAGC
4             XM_047432085.1   48    A      2  TGAGC
...                      ...  ...  ...    ...    ...
125397419  ENST00000804627.1  927    A      1  TCAGT
125397420  ENST00000804627.1  936    A      1  CCAGC
125397421  ENST00000804627.1  940    A      1  CCATT
125397422  ENST00000804627.1  946    A      1  CGACT
125397423  ENST00000804627.1  956    A      1  CCACG

[125397424 rows x 5 columns]


In [11]:
hek_label["ref"] = hek_label["ref"].apply(utils.reformat_transcript_id)
hela_label["ref"] = hela_label["ref"].apply(utils.reformat_transcript_id)

In [12]:
print(hek_label)

                       ref  pos base  depth   5mer
0           XM_047432085.1   12    A      1  CGACT
1           XM_047432085.1   28    A      2  GGAGG
2           XM_047432085.1   37    A      2  TGAGG
3           XM_047432085.1   42    A      2  CGAGC
4           XM_047432085.1   48    A      2  TGAGC
...                    ...  ...  ...    ...    ...
125397419  ENST00000804627  927    A      1  TCAGT
125397420  ENST00000804627  936    A      1  CCAGC
125397421  ENST00000804627  940    A      1  CCATT
125397422  ENST00000804627  946    A      1  CGACT
125397423  ENST00000804627  956    A      1  CCACG

[125397424 rows x 5 columns]


In [13]:
print(refflat_df)

                    transcript_id     gene_id   gene_name    chr strand  \
0                 ENST00000000233        ARF5  ARF5,FSCN3   chr7      +   
1                 ENST00000000442       ESRRA       ESRRA  chr11      +   
2                 ENST00000001008       FKBP4       FKBP4  chr12      +   
3                 ENST00000002125     NDUFAF7     NDUFAF7   chr2      +   
4                 ENST00000002165       FUCA2       FUCA2   chr6      -   
...                           ...         ...         ...    ...    ...   
542603  unassigned_transcript_995  TRH-GTG1-5  TRH-GTG1-5   chr6      +   
542604  unassigned_transcript_996  TRT-AGT6-1  TRT-AGT6-1   chr6      +   
542605  unassigned_transcript_997  TRI-AAT5-2  TRI-AAT5-2   chr6      -   
542606  unassigned_transcript_998  TRV-CAC6-1  TRV-CAC6-1   chr6      -   
542607  unassigned_transcript_999  TRS-CGA2-1  TRS-CGA2-1   chr6      +   

          txStart      txEnd   cdsStart     cdsEnd  exonCount  \
0       127588410  127591700  1275

In [14]:
coding_dict = dict(zip(refflat_df["transcript_id"], refflat_df["coding"]))

In [15]:
hek_label["coding"] = hek_label["ref"].apply(lambda x: coding_dict.get(x, False))
hela_label["coding"] = hela_label["ref"].apply(lambda x: coding_dict.get(x, False))

In [17]:
import itertools as it

In [18]:
all_possible_5mers = ["".join(x) for x in it.product("ACGT", repeat=5)]
all_possible_5mers = [x for x in all_possible_5mers if x[2] == "A"]
print(all_possible_5mers)

['AAAAA', 'AAAAC', 'AAAAG', 'AAAAT', 'AAACA', 'AAACC', 'AAACG', 'AAACT', 'AAAGA', 'AAAGC', 'AAAGG', 'AAAGT', 'AAATA', 'AAATC', 'AAATG', 'AAATT', 'ACAAA', 'ACAAC', 'ACAAG', 'ACAAT', 'ACACA', 'ACACC', 'ACACG', 'ACACT', 'ACAGA', 'ACAGC', 'ACAGG', 'ACAGT', 'ACATA', 'ACATC', 'ACATG', 'ACATT', 'AGAAA', 'AGAAC', 'AGAAG', 'AGAAT', 'AGACA', 'AGACC', 'AGACG', 'AGACT', 'AGAGA', 'AGAGC', 'AGAGG', 'AGAGT', 'AGATA', 'AGATC', 'AGATG', 'AGATT', 'ATAAA', 'ATAAC', 'ATAAG', 'ATAAT', 'ATACA', 'ATACC', 'ATACG', 'ATACT', 'ATAGA', 'ATAGC', 'ATAGG', 'ATAGT', 'ATATA', 'ATATC', 'ATATG', 'ATATT', 'CAAAA', 'CAAAC', 'CAAAG', 'CAAAT', 'CAACA', 'CAACC', 'CAACG', 'CAACT', 'CAAGA', 'CAAGC', 'CAAGG', 'CAAGT', 'CAATA', 'CAATC', 'CAATG', 'CAATT', 'CCAAA', 'CCAAC', 'CCAAG', 'CCAAT', 'CCACA', 'CCACC', 'CCACG', 'CCACT', 'CCAGA', 'CCAGC', 'CCAGG', 'CCAGT', 'CCATA', 'CCATC', 'CCATG', 'CCATT', 'CGAAA', 'CGAAC', 'CGAAG', 'CGAAT', 'CGACA', 'CGACC', 'CGACG', 'CGACT', 'CGAGA', 'CGAGC', 'CGAGG', 'CGAGT', 'CGATA', 'CGATC', 'CGATG', 

In [19]:
drach_dict = {x:utils.is_drach(x) for x in all_possible_5mers}
print(drach_dict)

{'AAAAA': False, 'AAAAC': False, 'AAAAG': False, 'AAAAT': False, 'AAACA': True, 'AAACC': True, 'AAACG': False, 'AAACT': True, 'AAAGA': False, 'AAAGC': False, 'AAAGG': False, 'AAAGT': False, 'AAATA': False, 'AAATC': False, 'AAATG': False, 'AAATT': False, 'ACAAA': False, 'ACAAC': False, 'ACAAG': False, 'ACAAT': False, 'ACACA': False, 'ACACC': False, 'ACACG': False, 'ACACT': False, 'ACAGA': False, 'ACAGC': False, 'ACAGG': False, 'ACAGT': False, 'ACATA': False, 'ACATC': False, 'ACATG': False, 'ACATT': False, 'AGAAA': False, 'AGAAC': False, 'AGAAG': False, 'AGAAT': False, 'AGACA': True, 'AGACC': True, 'AGACG': False, 'AGACT': True, 'AGAGA': False, 'AGAGC': False, 'AGAGG': False, 'AGAGT': False, 'AGATA': False, 'AGATC': False, 'AGATG': False, 'AGATT': False, 'ATAAA': False, 'ATAAC': False, 'ATAAG': False, 'ATAAT': False, 'ATACA': False, 'ATACC': False, 'ATACG': False, 'ATACT': False, 'ATAGA': False, 'ATAGC': False, 'ATAGG': False, 'ATAGT': False, 'ATATA': False, 'ATATC': False, 'ATATG': Fals

In [20]:
hek_label["drach"] = hek_label["5mer"].apply(lambda x: drach_dict.get(x, False))
hela_label["drach"] = hela_label["5mer"].apply(lambda x: drach_dict.get(x, False))

In [21]:
print(hek_label)
print(hela_label)

                       ref  pos base  depth   5mer  coding  drach
0           XM_047432085.1   12    A      1  CGACT    True  False
1           XM_047432085.1   28    A      2  GGAGG    True  False
2           XM_047432085.1   37    A      2  TGAGG    True  False
3           XM_047432085.1   42    A      2  CGAGC    True  False
4           XM_047432085.1   48    A      2  TGAGC    True  False
...                    ...  ...  ...    ...    ...     ...    ...
125397419  ENST00000804627  927    A      1  TCAGT   False  False
125397420  ENST00000804627  936    A      1  CCAGC   False  False
125397421  ENST00000804627  940    A      1  CCATT   False  False
125397422  ENST00000804627  946    A      1  CGACT   False  False
125397423  ENST00000804627  956    A      1  CCACG   False  False

[125397424 rows x 7 columns]
                       ref  pos base  depth   5mer  coding  drach
0          ENST00000260843    4    A      4  GTAGT    True  False
1          ENST00000260843   15    A      5  G

In [23]:
hek_label["label_id"] = hek_label["ref"] + ":" + (hek_label["pos"]-1).astype(str)
hela_label["label_id"] = hela_label["ref"] + ":" + (hela_label["pos"]-1).astype(str)

In [27]:
hek_label.rename(columns={"ref": "transcript_id"}, inplace=True)
hela_label.rename(columns={"ref": "transcript_id"}, inplace=True)

geneid_table = refflat_df[["transcript_id", "gene_id", "chr", "strand"]]

In [28]:
hek_label = hek_label.merge(geneid_table, how="left", on="transcript_id")
hela_label = hela_label.merge(geneid_table, how="left", on="transcript_id")

In [29]:
print(hek_label)
print(hela_label)

             transcript_id  pos base  depth   5mer  coding  drach  \
0           XM_047432085.1   12    A      1  CGACT    True  False   
1           XM_047432085.1   28    A      2  GGAGG    True  False   
2           XM_047432085.1   37    A      2  TGAGG    True  False   
3           XM_047432085.1   42    A      2  CGAGC    True  False   
4           XM_047432085.1   48    A      2  TGAGC    True  False   
...                    ...  ...  ...    ...    ...     ...    ...   
125397419  ENST00000804627  927    A      1  TCAGT   False  False   
125397420  ENST00000804627  936    A      1  CCAGC   False  False   
125397421  ENST00000804627  940    A      1  CCATT   False  False   
125397422  ENST00000804627  946    A      1  CGACT   False  False   
125397423  ENST00000804627  956    A      1  CCACG   False  False   

                      label_id       gene_id   chr strand  
0            XM_047432085.1:11         ACBD6  chr1      -  
1            XM_047432085.1:27         ACBD6  chr1 

In [30]:
hek_label.to_pickle("/extdata4/baeklab/Hyeonseo/m6A/runs/exp_MRNA/ON0090/ON0090/result/reference_test/dorado_output.pileup.label.pkl")
hela_label.to_pickle("/extdata4/baeklab/Hyeonseo/m6A/runs/exp_MRNA/ON0091/ON0091/result/reference_test/dorado_output.pileup.label.pkl")